# PJM Load Forecast — DA Cutoff Revision Analysis (v2)

Inspects how PJM load forecasts for the next 7 days evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 12h / 24h / 48h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates and regions.

## 1. Setup & Data Pull

In [33]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

In [34]:
# Load the main cutoff analysis query
sql_path = Path.cwd() / "pjm_load_forecast_da_cutoff.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"Regions: {sorted(df['region'].unique())}")
df.head(10)

1,248 rows
Date range: 2026-02-25 to 2026-03-10
Regions: ['MIDATL', 'RTO', 'SOUTH', 'WEST']


,forecast_date,hour_ending,region,cutoff_load_mw,cutoff_execution_dt,exec_dt_12h,load_mw_12h,exec_dt_24h,load_mw_24h,exec_dt_48h,load_mw_48h,delta_12h,delta_24h,delta_48h
0,2026-03-10,1,MIDATL,22970.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
1,2026-03-10,2,MIDATL,22199.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
2,2026-03-10,3,MIDATL,21784.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
3,2026-03-10,4,MIDATL,21665.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
4,2026-03-10,5,MIDATL,22226.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
5,2026-03-10,6,MIDATL,23719.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
6,2026-03-10,7,MIDATL,26164.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
7,2026-03-10,8,MIDATL,27595.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
8,2026-03-10,9,MIDATL,27678.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
9,2026-03-10,10,MIDATL,27011.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN


## 2. Data Validation — Cutoff Vintage Inspection

In [36]:
# Actual cutoff execution timestamps per forecast_date
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-02-25,2026-02-25 08:47:27,08:47
1,2026-02-26,2026-02-26 08:47:10,08:47
2,2026-02-27,2026-02-27 08:47:25,08:47
3,2026-02-28,2026-02-28 08:47:10,08:47
4,2026-03-01,2026-03-01 08:47:24,08:47
5,2026-03-02,2026-03-02 08:47:09,08:47
6,2026-03-03,2026-03-03 08:47:32,08:47
7,2026-03-04,2026-03-04 08:47:16,08:47
8,2026-03-05,2026-03-04 08:47:16,08:47
9,2026-03-06,2026-03-04 08:47:16,08:47


In [37]:
# Bar chart of cutoff hour-of-day
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [38]:
# Lookback coverage: which dates have 12h/24h/48h data
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("load_mw_12h", lambda x: x.notna().any()),
        has_24h=("load_mw_24h", lambda x: x.notna().any()),
        has_48h=("load_mw_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-02-25,False,False,False,NaT,NaT,NaT
1,2026-02-26,True,True,False,2026-02-25 20:17:33,2026-02-25 08:17:27,NaT
2,2026-02-27,True,True,True,2026-02-26 20:47:17,2026-02-26 08:47:10,2026-02-25 08:17:27
3,2026-02-28,True,True,True,2026-02-27 13:47:28,2026-02-27 08:17:24,2026-02-26 08:47:10
4,2026-03-01,True,True,True,2026-02-28 10:47:11,2026-02-28 08:47:10,2026-02-27 08:17:24
5,2026-03-02,True,True,True,2026-03-01 10:47:25,2026-03-01 08:17:24,2026-02-28 08:17:10
6,2026-03-03,True,True,True,2026-03-02 12:47:12,2026-03-02 08:47:09,2026-03-01 08:47:24
7,2026-03-04,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
8,2026-03-05,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
9,2026-03-06,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [39]:
# For each region, overlay 48h → 24h → 12h → cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].copy()

regions = sorted(df_latest["region"].unique())
fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r} — {latest_date}" for r in regions],
    vertical_spacing=0.06,
)

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

for i, region in enumerate(regions, 1):
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    for label, col, width in [
        ("48h", "load_mw_48h", 1),
        ("24h", "load_mw_24h", 1.5),
        ("12h", "load_mw_12h", 2),
        ("Cutoff", "cutoff_load_mw", 3),
    ]:
        fig.add_trace(
            go.Scatter(
                x=rdf["hour_ending"],
                y=rdf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=300 * len(regions),
    template="plotly_dark",
    title_text=f"Forecast Evolution — Latest Date ({latest_date})",
)
fig.update_yaxes(title_text="Load (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(regions), col=1)
fig.show()

In [40]:
# Same view for all dates — one chart per region, lines per forecast_date
for region in regions:
    rdf = df[df["region"] == region].copy()
    dates = sorted(rdf["forecast_date"].unique())

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=[f"{region} — Cutoff vs 24h Lookback"],
    )

    for j, d in enumerate(dates):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")
        opacity = 0.3 if d != latest_date else 1.0
        # Cutoff
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["cutoff_load_mw"],
            name=f"{d} cutoff",
            line=dict(width=2 if d == latest_date else 1),
            opacity=opacity,
            showlegend=True,
        ))
        # 24h lookback (dashed)
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["load_mw_24h"],
            name=f"{d} 24h",
            line=dict(dash="dash", width=1),
            opacity=opacity * 0.6,
            showlegend=False,
        ))

    fig.update_layout(
        height=500, template="plotly_dark",
        title=f"{region} — Cutoff (solid) vs 24h Prior (dashed)",
        xaxis_title="Hour Ending", yaxis_title="Load (MW)",
    )
    fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [41]:
# Grouped bar charts: delta_12h, delta_24h, delta_48h by hour_ending for latest date
for region in regions:
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    fig = go.Figure()
    for col, label, color in [
        ("delta_48h", "48h\u2192Cutoff", "#636EFA"),
        ("delta_24h", "24h\u2192Cutoff", "#EF553B"),
        ("delta_12h", "12h\u2192Cutoff", "#00CC96"),
    ]:
        fig.add_trace(go.Bar(
            x=rdf["hour_ending"], y=rdf[col],
            name=label, marker_color=color,
        ))

    fig.update_layout(
        barmode="group",
        title=f"{region} — Forecast Revision Deltas ({latest_date})",
        xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
        template="plotly_dark", height=400,
    )
    fig.add_hline(y=0, line_color="white", line_width=0.5)
    fig.show()

In [42]:
# Heatmap: 24h delta across all dates × hours for RTO
rto = df[df["region"] == "RTO"].copy()
pivot_24h = rto.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_24h",
)

fig = px.imshow(
    pivot_24h.values,
    x=[f"HE{int(c)}" for c in pivot_24h.columns],
    y=[str(d) for d in pivot_24h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="RTO — 24h Forecast Revision (MW) by Date \u00d7 Hour",
    labels={"color": "Delta MW"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. Per-Date Revision Deep Dive

For each forecast date, show the 48h → 24h → 12h → cutoff evolution per region
so we can inspect how each day's forecast converged.

In [43]:
# One subplot row per date, one chart per region
dates = sorted(df["forecast_date"].unique())

for region in regions:
    rdf = df[df["region"] == region].copy()

    fig = make_subplots(
        rows=len(dates), cols=1,
        shared_xaxes=True,
        subplot_titles=[f"{region} — {d}" for d in dates],
        vertical_spacing=0.04,
    )

    for i, d in enumerate(dates, 1):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")

        for label, col, width in [
            ("48h", "load_mw_48h", 1),
            ("24h", "load_mw_24h", 1.5),
            ("12h", "load_mw_12h", 2),
            ("Cutoff", "cutoff_load_mw", 3),
        ]:
            fig.add_trace(
                go.Scatter(
                    x=ddf["hour_ending"],
                    y=ddf[col],
                    name=label,
                    line=dict(color=colors[label], dash=dashes[label], width=width),
                    showlegend=(i == 1),
                ),
                row=i, col=1,
            )

    fig.update_layout(
        height=250 * len(dates),
        template="plotly_dark",
        title_text=f"{region} — Forecast Evolution by Date",
    )
    fig.update_yaxes(title_text="Load (MW)")
    fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
    fig.show()

## 6. Revision Summary Statistics

In [44]:
# Summary table: mean, median, std of deltas by region and lookback
summary_rows = []
for region in regions:
    rdf = df[df["region"] == region]
    for lookback, col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
        vals = rdf[col].dropna()
        summary_rows.append({
            "Region": region,
            "Lookback": lookback,
            "Mean (MW)": vals.mean(),
            "Median (MW)": vals.median(),
            "Std (MW)": vals.std(),
            "Min (MW)": vals.min(),
            "Max (MW)": vals.max(),
            "N": len(vals),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Region,Lookback,Mean (MW),Median (MW),Std (MW),Min (MW),Max (MW),N
0,MIDATL,12h,234.734848,0.0,655.742384,-1912.0,1584.0,264
1,MIDATL,24h,282.393939,203.0,714.599973,-1745.0,1717.0,264
2,MIDATL,48h,374.407407,446.0,1002.597311,-2446.0,2283.0,216
3,RTO,12h,130.924242,127.5,1008.451379,-3204.0,2715.0,264
4,RTO,24h,237.458333,174.0,1129.574706,-2406.0,2861.0,264
5,RTO,48h,443.050926,623.0,1612.211987,-4237.0,3321.0,216
6,SOUTH,12h,3.545455,0.0,277.167459,-752.0,880.0,264
7,SOUTH,24h,-2.015152,-41.5,306.773842,-723.0,856.0,264
8,SOUTH,48h,102.287037,45.5,365.489199,-771.0,944.0,216
9,WEST,12h,-107.356061,-24.0,470.688156,-2185.0,1025.0,264


In [45]:
# Per-date summary: avg revision direction across all hours
date_summary = (
    df.groupby(["forecast_date", "region"])
    .agg(
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        mean_delta_48h=("delta_48h", "mean"),
        peak_cutoff_mw=("cutoff_load_mw", "max"),
    )
    .reset_index()
)

# Grouped bar: mean 24h delta per date, faceted by region
fig = px.bar(
    date_summary,
    x="forecast_date",
    y="mean_delta_24h",
    color="region",
    barmode="group",
    title="Mean 24h Forecast Revision (MW) by Date & Region",
    labels={"mean_delta_24h": "Avg 24h Delta (MW)", "forecast_date": "Forecast Date"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.update_layout(height=450)
fig.show()

In [46]:
# Heatmaps for all regions (12h, 24h, 48h)
for lookback, delta_col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    for region in regions:
        rdf = df[df["region"] == region].copy()
        pivot = rdf.pivot_table(
            index="forecast_date", columns="hour_ending", values=delta_col,
        )
        if pivot.empty:
            continue

        fig = px.imshow(
            pivot.values,
            x=[f"HE{int(c)}" for c in pivot.columns],
            y=[str(d) for d in pivot.index],
            color_continuous_scale="RdBu",
            color_continuous_midpoint=0,
            aspect="auto",
            title=f"{region} — {lookback} Forecast Revision (MW) by Date \u00d7 Hour",
            labels={"color": "Delta MW"},
            template="plotly_dark",
        )
        fig.update_layout(height=400)
        fig.show()